## 1- Import data frames

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

In [ ]:
result= pd.read_csv("../clean_data/result_for_data_cleaning.csv") 
outcome1 =pd.read_csv("../original_data/outcomes\Outcomes-a.txt") 
outcome2 =pd.read_csv("../original_data/outcomes\Outcomes-b.txt") 

## 2- Merge the patients information with the outcomes

In [ ]:
pd.options.display.max_columns = None
outcome1

In [ ]:
# concat all the outcomes and then merge with result so we have all the patient data as well as the outcome
outcome = pd.concat([outcome1, outcome2], axis=0, ignore_index=True)
final_df = pd.merge(result, outcome, on='RecordID')

In [ ]:
final_df.head()

## 3- Drop any patient not holding medical informartion

In [ ]:
# drop the patients that do not hold any medical value 
non_medical_values = ['RecordID', 'Age'	,'Gender','Height', 'SAPS-I',	'SOFA',	'Length_of_stay','Survival','In-hospital_death']
final_df.dropna(subset=[col for col in final_df.columns if col not in non_medical_values], how='all', inplace=True)

In [ ]:
final_df.shape # no lines have been delated

## 4- NaN imputation

In [ ]:
# drop num_samples columns
columns_with_number_samples = [x for x in final_df.columns if '_num_samples' in  x.lower()]
final_df.drop(columns=columns_with_number_samples, axis= 1, inplace= True)

In [ ]:
final_df.isna().sum()

4.1- drop those patientes with NaN in Gender column

In [ ]:
final_df.dropna(subset=['Gender'], inplace=True)

4.1- Steps for imputing numerial values to Nan:

- Select the columns without NaN values to use for clustering.

- Perform clustering on these selected columns to group patients into clusters.

- Calculate the mean of each cluster.

- Assign the mean value of the cluster to NaN values for each patient in the same cluster.

In [ ]:
def impute_nan_with_cluster_mean(df, n_clusters=5):
    # Set 'RecordID' as index
    df.set_index('RecordID', inplace=True)
    
    # Select columns without NaN values
    columns_without_nan = df.columns[df.isnull().sum() == 0]
    data_without_nan = df[columns_without_nan]
    
    # Perform clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(data_without_nan)
    
    # Assign cluster labels to DataFrame
    df['Cluster'] = clusters
    
    # Calculate cluster means
    cluster_means = df.groupby('Cluster').mean()
    
    # Impute NaN values with cluster means
    for index, row in df.iterrows():
        cluster = row['Cluster']
        for column in df.columns:
            if pd.isna(row[column]):
                df.at[index, column] = cluster_means.at[cluster, column]
    
    return df

In [ ]:
final_df = impute_nan_with_cluster_mean(final_df)

In [ ]:
final_df.sample(5)

## 5- Logistic Regression for In-hospital_death Prediction and Linear regression for Length_of_stay prediction